In [1]:
import pandas as pd

# Priprema podataka (samo stance klasifikacija)

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.metrics import classification_report, f1_score

# ---------------------------------------------------------
# 1) Ucitavanje i priprema podataka
# ---------------------------------------------------------
df = pd.read_csv("novi_final_20k.csv")

# baciti redove bez teksta (nema sta trenirati)
df = df.dropna(subset=["SADRZAJ"]).reset_index(drop=True)

TOPICS = [
    "euroatlantske_integracije",
    "negiranje_genocida",
    "gradjanska_vs_konstitutivni",
    "izborna_reforma",
]

def stance_label(row, topic):
    """stance -> against / neutral / for (bez not_mentioned, koristi se samo na mentioned redovima)"""
    stance = row[f"{topic}_stance"]
    if stance == 1:
        return "for"
    elif stance == -1:
        return "against"
    else:
        return "neutral"

# Za svaku temu napravimo poseban dataframe koji sadrzi SAMO redove
# gdje je tema zaista spomenuta (mentioned == 1/True). Redovi gdje
# tema NIJE spomenuta se u potpunosti izbacuju - ne treniramo i ne
# evaluiramo na not_mentioned klasi.
df_topic_stance = {}
for topic in TOPICS:
    mask = df[f"{topic}_mentioned"].astype(bool)
    d = df.loc[mask].copy()
    d[f"{topic}_label"] = d.apply(lambda r: stance_label(r, topic), axis=1)
    df_topic_stance[topic] = d
    print(f"{topic}: {mask.sum()} od {len(df)} redova je 'mentioned' -> koristi se za stance klasifikaciju")

# ---------------------------------------------------------
# 2) TF-IDF (word + char_wb n-grami) preko FeatureUnion
# ---------------------------------------------------------
def build_vectorizer():
    return FeatureUnion([
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.9,
            sublinear_tf=True,
        )),
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_df=0.9,
            sublinear_tf=True,
        )),
    ])

euroatlantske_integracije: 6287 od 19820 redova je 'mentioned' -> koristi se za stance klasifikaciju
negiranje_genocida: 3485 od 19820 redova je 'mentioned' -> koristi se za stance klasifikaciju
gradjanska_vs_konstitutivni: 5292 od 19820 redova je 'mentioned' -> koristi se za stance klasifikaciju
izborna_reforma: 3027 od 19820 redova je 'mentioned' -> koristi se za stance klasifikaciju


# Dio 1: TF-IDF + SGD / Logistic Regression (baseline, bez grid searcha)

Osnovni baseline: TF-IDF vektorizacija + `LogisticRegression` i `SGDClassifier`
(sa hinge i log_loss gubitkom) sa fiksnim, razumnim hiperparametrima.

Klasifikacija je sada **samo stance** (`for` / `against` / `neutral`) i radi se
isključivo na redovima gdje je tema spomenuta - `not_mentioned` redovi su
izbačeni iz treninga i evaluacije za tu temu.

In [3]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.base import clone

MODELS_PART1 = {
    "logreg": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=1.0,
        solver="saga",
        n_jobs=-1,
    ),
    "sgd_hinge": SGDClassifier(
        loss="hinge",
        class_weight="balanced",
        alpha=1e-5,
        max_iter=2000,
        random_state=42,
    ),
    "sgd_log": SGDClassifier(
        loss="log_loss",
        class_weight="balanced",
        alpha=1e-5,
        max_iter=2000,
        random_state=42,
    ),
}

In [ ]:
# ---------------------------------------------------------
# Treniranje posebnog stance modela po temi (Dio 1 - baseline)
# ---------------------------------------------------------
results_part1 = {}

for topic in TOPICS:
    print("=" * 70)
    print(f"TEMA: {topic} (STANCE)")
    print("=" * 70)

    d = df_topic_stance[topic]
    X = d["SADRZAJ"]
    y = d[f"{topic}_label"]

    print("Distribucija klasa:\n", y.value_counts(), "\n")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    for model_name, clf in MODELS_PART1.items():
        pipe = Pipeline([
            ("tfidf", build_vectorizer()),
            ("clf", clone(clf)),
        ])

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

        print(f"--- {model_name} | macro-F1 = {f1_macro:.3f} ---")
        print(classification_report(y_test, y_pred, zero_division=0))

        results_part1[(topic, model_name)] = {
            "pipeline": pipe,
            "f1_macro": f1_macro,
        }

In [5]:
import joblib
import os

os.makedirs("models_stance", exist_ok=True)

for (topic, model_name), res in results_part1.items():
    path = f"models_stance/{topic}__{model_name}_stance.joblib"
    joblib.dump(res["pipeline"], path)
    print(f"Sacuvano: {path} (f1_macro={res['f1_macro']:.3f})")

Sacuvano: models_stance/euroatlantske_integracije__logreg_stance.joblib (f1_macro=0.729)
Sacuvano: models_stance/euroatlantske_integracije__sgd_hinge_stance.joblib (f1_macro=0.699)
Sacuvano: models_stance/euroatlantske_integracije__sgd_log_stance.joblib (f1_macro=0.714)
Sacuvano: models_stance/negiranje_genocida__logreg_stance.joblib (f1_macro=0.708)
Sacuvano: models_stance/negiranje_genocida__sgd_hinge_stance.joblib (f1_macro=0.699)
Sacuvano: models_stance/negiranje_genocida__sgd_log_stance.joblib (f1_macro=0.699)
Sacuvano: models_stance/gradjanska_vs_konstitutivni__logreg_stance.joblib (f1_macro=0.712)
Sacuvano: models_stance/gradjanska_vs_konstitutivni__sgd_hinge_stance.joblib (f1_macro=0.707)
Sacuvano: models_stance/gradjanska_vs_konstitutivni__sgd_log_stance.joblib (f1_macro=0.724)
Sacuvano: models_stance/izborna_reforma__logreg_stance.joblib (f1_macro=0.736)
Sacuvano: models_stance/izborna_reforma__sgd_hinge_stance.joblib (f1_macro=0.722)
Sacuvano: models_stance/izborna_reforma__

In [6]:
# ---------------------------------------------------------
# Pregled rezultata - Dio 1 (stance)
# ---------------------------------------------------------
summary_part1 = pd.DataFrame([
    {"tema": t, "model": m, "macro_f1": r["f1_macro"]}
    for (t, m), r in results_part1.items()
]).sort_values(["tema", "macro_f1"], ascending=[True, False])

print(summary_part1.to_string(index=False))

                       tema     model  macro_f1
  euroatlantske_integracije    logreg  0.729418
  euroatlantske_integracije   sgd_log  0.714420
  euroatlantske_integracije sgd_hinge  0.699387
gradjanska_vs_konstitutivni   sgd_log  0.723817
gradjanska_vs_konstitutivni    logreg  0.711654
gradjanska_vs_konstitutivni sgd_hinge  0.707081
            izborna_reforma    logreg  0.736053
            izborna_reforma   sgd_log  0.731522
            izborna_reforma sgd_hinge  0.721643
         negiranje_genocida    logreg  0.708259
         negiranje_genocida   sgd_log  0.699187
         negiranje_genocida sgd_hinge  0.698952


# Inferenca

Učitavanje sačuvanih stance modela i predikcija na novom tekstu.
Za svaku temu se dobija jedna od tri klase: `for`, `against`, `neutral`.
Napomena: ovi modeli ne znaju prepoznati da tema uopšte NIJE spomenuta -
oni uvijek vraćaju stav (for/against/neutral), pa ih ima smisla koristiti
samo nad tekstovima za koje se zna (ili se pretpostavlja) da tema jeste
spomenuta.

In [ ]:
euro_m = joblib.load("models_stance/euroatlantske_integracije__logreg_stance.joblib")
izborna_m = joblib.load("models_stance/izborna_reforma__logreg_stance.joblib")
genocid_m = joblib.load("models_stance/negiranje_genocida__logreg_stance.joblib")
grad_m = joblib.load("models_stance/gradjanska_vs_konstitutivni__logreg_stance.joblib")

In [ ]:
def predict_stance(text):
    euro_p = euro_m.predict(text)[0]
    izborna_p = izborna_m.predict(text)[0]
    genocid_p = genocid_m.predict(text)[0]
    grad_p = grad_m.predict(text)[0]
    return {"izborna_p": izborna_p, "euro_p": euro_p, "genocid_p": genocid_p, "grad_p": grad_p}

In [ ]:
text = """
Ovdje idi svoj tekst za testiranje.
"""

In [ ]:
predict_stance([text])